# Suite PRS — Parsing

The C++ standards parse source text into Ccpp trees by delegating to tree-sitter-cpp, then rewriting its concrete
syntax tree into Ccpp kinds; a token-level pre-pass handles the module declarations, `export` and `extern template`
that tree-sitter-cpp 0.23 cannot parse. Each case parses constructs and checks the tree (by shape where it matters,
and by the text it prints), then the errors: syntax errors, unsupported syntax, and constructs a standard lacks,
each with its line and column.

In [ ]:
from pathlib import Path

from mbse.Programs.Ccpp import Ccpp17, Ccpp20, Syntax as S
from mbse.Programs.Ccpp._Standard import CcppStandard
from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Framework.Errors import ParseError
from support import raises

CPP26, C23 = CcppStandard(2026), CcppStandard(2023, "C")
SOURCES = Path("../../conformance/sources")


def parse(text, standard=CPP26):
    unit = standard.parse(text)
    assert S.LANGUAGE.validate(unit) == [], S.LANGUAGE.validate(unit)
    return unit


def first(text, standard=CPP26):
    return parse(text, standard).items[0]


def same_text(text, expected=None, standard=CPP26):
    """Parses `text`, checks it prints `expected` (by default itself) and that the print parses to the same tree."""
    unit = parse(text, standard)
    printed = standard.print(unit)
    assert printed == (text if expected is None else expected) + "\n", printed
    assert F.same(parse(printed, standard), unit)
    return unit


def same_statement(text, expected=None, standard=CPP26):
    """`same_text` for a statement, in a function's body."""
    def wrap(t):
        return "void f() {\n    " + t.replace("\n", "\n    ") + "\n}"
    return same_text(wrap(text), wrap(expected if expected is not None else text), standard).items[0].body.items[0]


def initializer(text, standard=CPP26):
    """The value initializing the one variable `text` declares."""
    return first(text, standard).declarators[0].initializer.value


def fails(text, message, line, column, standard=CPP26):
    with raises(ParseError, match=message):
        standard.parse(text)
    try:
        standard.parse(text)
    except ParseError as error:
        assert (error.line, error.column) == (line, column), (error.line, error.column)
        assert str(error) == f"line {line}, column {column}: {message}", str(error)

## PRS-01 · The conformance source parses to a valid tree, which prints and parses back the same

In [ ]:
source = (SOURCES / "cpp20.cpp").read_text(encoding="utf-8")
unit = Ccpp20.parse(source)
assert S.LANGUAGE.validate(unit) == []
assert F.same(Ccpp20.parse(Ccpp20.print(unit)), unit) and Ccpp20.print(Ccpp20.parse(Ccpp20.print(unit))) == Ccpp20.print(unit)
kinds = {node.KIND for node in F.walk(unit)}
assert len(kinds) > 120, len(kinds)
print(len(kinds), "kinds")

## PRS-02 · Names: qualified, template, operator, conversion, destructor and literal operator names

In [ ]:
name = first("a::b<int>::template c<T>::d x;").specifiers[0].name
assert isinstance(name, S.QualifiedName) and not name.global_scope and name.name.spelling == "d"
assert [type(q).__name__ for q in name.qualifiers] == ["Identifier", "TemplateId", "TemplateId"]
assert not name.qualifiers[1].template_keyword and name.qualifiers[2].template_keyword
name = first("A::template B<int> x;").specifiers[0].name
assert name.name.template_keyword and name.name.name.spelling == "B"
assert first("::x y;").specifiers[0].name.global_scope
scope = first("decltype(x)::type y;").specifiers[0].name.qualifiers[0]
assert isinstance(scope, S.DecltypeSpecifier)
same_text("auto x = A::template f<int>();")
same_text("X<Ts...> x;")
assert isinstance(first("X<Ts...> x;").specifiers[0].name.arguments[0].pattern, S.TypeId)
operators = first("struct S { S &operator=(S &&); void *operator new[](size_t); bool operator<=>(S) const; "
                  "void operator()(); auto operator co_await(); bool operator not_eq(S); };").specifiers[0].body.items
assert [n.operator for d in operators for n in F.walk(d) if isinstance(n, S.OperatorName)] == [
    "=", "new[]", "<=>", "()", "co_await", "!="]
literal = first('int operator""_km(unsigned long long);').declarators[0].declarator.declarator.name
assert isinstance(literal, S.LiteralOperatorName) and literal.suffix == "_km"
destructor = first("A::~A() {}").declarator.declarator.name
assert isinstance(destructor, S.QualifiedName) and isinstance(destructor.name, S.DestructorName)
same_text("struct S {\n    operator int&() const;\n    operator int&&();\n    explicit operator const char*() const;\n};")
same_text("A::operator int*() const {}\n::A::B::operator bool() {}")
qualified = first("A::operator int*() const {}").declarator.declarator.name
assert isinstance(qualified, S.QualifiedName) and isinstance(qualified.name, S.ConversionName)
conversion = first("operator int*() const;").declarators[0].declarator
assert isinstance(conversion.declarator.name.type.declarator, S.PointerDeclarator) and conversion.qualifiers[0].keyword == "const"
print("ok")

## PRS-03 · Literals keep their spelling

In [ ]:
values = first("auto a = {1'000ull, 0x1.8p3, 1.5e-3, 0b101, 0x1F, 'c', L'w', u8'x', \"s\\n\", R\"d(r)d\", "
               "u8R\"(x)\", 1_km, \"s\"_sv, true, false, nullptr, NULL};").declarators[0].initializer.value.items
assert [type(v).__name__ for v in values] == [
    "IntegerLiteral", "FloatingLiteral", "FloatingLiteral", "IntegerLiteral", "IntegerLiteral", "CharacterLiteral",
    "CharacterLiteral", "CharacterLiteral", "StringLiteral", "RawStringLiteral", "RawStringLiteral",
    "UserDefinedLiteral", "UserDefinedLiteral", "BooleanLiteral", "BooleanLiteral", "NullptrLiteral", "IdExpression"]
assert values[0].spelling == "1'000ull" and values[6].prefix == "L" and values[7].prefix == "u8"
assert values[8].text == "s\\n" and values[8].prefix is None
assert (values[9].prefix, values[9].delimiter, values[9].text) == (None, "d", "r")
assert (values[10].prefix, values[10].delimiter) == ("u8", None) and values[11].suffix == "_km"
assert values[13].value is True and values[14].value is False
concatenated = initializer('auto s = "a" L"b" PRIu64 R"(c)";')
assert [type(p).__name__ for p in concatenated.parts] == ["StringLiteral", "StringLiteral", "IdExpression",
                                                          "RawStringLiteral"]
same_text('auto s = "a" L"b" PRIu64 R"(c)";')
print("ok")

## PRS-04 · Expressions: operators, alternative tokens, casts, calls, members, new and delete

In [ ]:
same_statement("x = a, b, c;")
comma = same_statement("x, y, z;").expression
assert comma.operator == "," and comma.left.operator == "," and comma.right.name.spelling == "z"  # to the left
assert initializer("bool b = a and not c or d bitand e;").right.operator == "&"
same_text("bool b = a and not c or d bitand e xor f bitor compl g;", "bool b = a && !c || d & e ^ f | ~g;")
same_statement("x and_eq 1, x or_eq 2, x xor_eq 3, x not_eq 4;", "x &= 1, x |= 2, x ^= 3, x != 4;")
same_statement("x = -a + +b * !c / ~d % *p - &q;")
same_statement("x = ++a + --b - a++ - b--;")
assert isinstance(initializer("int x = ++a;"), S.UnaryExpression)
assert isinstance(initializer("int x = a++;"), S.PostfixExpression)
same_statement("x = a ? b : c;")
same_statement("x = a ?: c;")
assert initializer("int x = a ?: c;").consequence is None
same_statement("x = (int)y + static_cast<int>(y) + int(y) + int{y} + T{y} + unsigned(y);")
assert isinstance(initializer("int x = int(y);"), S.FunctionalCastExpression)
assert isinstance(initializer("int x = f<int>(y);"), S.CallExpression)
assert isinstance(initializer("int x = static_cast<int, int>(y);"), S.CallExpression)  # not a cast
same_statement("x = dynamic_cast<B*>(p), const_cast<char*>(s), reinterpret_cast<long>(p);")
same_statement("x = f(1, g(2), h<int>(3)), a[1], m[1, 2], o.m, p->m, o.*f, s.template get<0>(), p->~T();")
same_statement("x = sizeof y + sizeof(y) + sizeof(int) + alignof(int) + __alignof__(int) + sizeof...(Ts);")
same_statement("x = noexcept(f()) && typeid(y) == typeid(*p);")
assert isinstance(initializer("bool b = noexcept(f());"), S.NoexceptExpression)
assert isinstance(initializer("auto &t = typeid(y);"), S.TypeidExpression)
fails("auto &t = typeid(y, z);", "typeid takes one operand", 1, 17)
same_statement("x = new T, new T(1), new T{1}, new int[n][4](), ::new (p) T, new (std::nothrow) T[2]{1, 2};")
new = initializer("int *x = new int[n][4]{};")
assert new.type.declarator.size.spelling == "4" and new.type.declarator.declarator.size.name.spelling == "n"
same_statement("delete p, delete[] q, ::delete r;")
same_statement("x = co_await y;")
same_statement("x = ({\n    int y = 1;\n    y;\n});")
assert isinstance(initializer("int x = ({ 1; });"), S.StatementExpression)
same_statement("x = __extension__ 1;")
same_statement("x = (... + xs) + (xs * ... * 1) + (xs && ...);")
same_statement("f(xs...), g(std::forward<Ts>(xs)...);")
same_statement("x = offsetof(S, m);")
assert isinstance(initializer("int x = offsetof(S, m);").arguments[0], S.TypeId)
same_statement("x = {1, {2, 3}, .a = 4};")
same_text("std::vector<int> v = {1, 2,};")
same_text("int x = defined;")
print("ok")

## PRS-05 · Lambdas, captures, fold expressions and requires-expressions

In [ ]:
lambda_ = initializer("auto f = [=, &a, b, this, *this, c = 1, &d = e, ...f = g, &...h = i, j..., &k...]"
                      "<typename T> requires C<T> (T t) mutable constexpr noexcept -> T requires D<T> "
                      "{ return t; };")
assert [type(c).__name__ for c in lambda_.captures] == [
    "DefaultCapture", "SimpleCapture", "SimpleCapture", "ThisCapture", "ThisCapture", "InitCapture", "InitCapture",
    "InitCapture", "InitCapture", "SimpleCapture", "SimpleCapture"]
assert lambda_.captures[1].by_reference and not lambda_.captures[2].by_reference and lambda_.captures[4].copy
assert lambda_.captures[6].by_reference and lambda_.captures[7].pack and lambda_.captures[8].by_reference
assert lambda_.captures[9].pack and lambda_.captures[10].pack and lambda_.captures[10].by_reference
declarator = lambda_.declarator
assert [s.keyword for s in declarator.specifiers] == ["mutable", "constexpr"]
assert isinstance(declarator.exception, S.NoexceptSpecifier) and declarator.requires is not None
same_text("auto f = [=, &a, b, this, *this, c = 1, &d = e, ...f = g, &...h = i, j..., &k...]<typename T> requires C<T>"
          " (T t) mutable constexpr noexcept -> T requires D<T> {\n    return t;\n};")
same_text("auto f = [] {};")
same_text("auto f = []() throw() [[nodiscard]] {};", standard=Ccpp17)
same_text("auto f = [](auto... x) {\n    return (x + ...);\n};")
with raises(ParseError, match="unsupported syntax: ref_qualifier"):
    CPP26.parse("auto f = []() & {};")
requires = first("template <class T> concept C = requires (T t, const T &u) { t.f(); typename T::x; "
                 "{ t.g() } noexcept -> std::same_as<int>; { u.h() }; requires D<T>; };").declaration.constraint
assert [type(r).__name__ for r in requires.requirements] == [
    "SimpleRequirement", "TypeRequirement", "CompoundRequirement", "CompoundRequirement", "NestedRequirement"]
assert requires.requirements[2].noexcept and requires.requirements[3].return_type is None
same_text("template <class T>\nconcept C = requires { T::value; } && (sizeof(T) > 1 || D<T>);")
same_text("template <class T> requires A<T> && (B<T> || C<T>) && D\nvoid f(T);")
print("ok")

## PRS-06 · Declarations and specifiers, in source order

In [ ]:
declaration = first("[[nodiscard]] static inline const unsigned long long int [[a]] *x = nullptr, y{1}, z(2);")
assert [type(a).__name__ for a in declaration.attributes] == ["StandardAttributeSpecifier"]
assert [s.keyword if hasattr(s, "keyword") else type(s).__name__ for s in declaration.specifiers] == [
    "static", "inline", "const", "unsigned", "long", "long", "int", "StandardAttributeSpecifier"]
assert [type(d.initializer).__name__ for d in declaration.declarators] == [
    "EqualInitializer", "InitializerList", "ParenthesizedInitializer"]
same_text("int const volatile x = 1;")
same_text("size_t n;")
assert isinstance(first("size_t n;").specifiers[0], S.NamedTypeSpecifier)
assert isinstance(first("wchar_t c;").specifiers[0], S.PrimitiveTypeSpecifier)
same_text("typedef int T, *P;")
same_text("__extension__ typedef long long ll;")
same_text("using T = typename A::B;")
same_text("extern thread_local int t;")
same_text("constexpr auto x = 1;\nconstinit int y = 2;\nconsteval int f();")
same_text("decltype(auto) x = y;\nC auto z = w;\nC<int> decltype(auto) v = u;")
same_text("struct S {\n    explicit(true) S();\n};")
same_text("__declspec(dllexport) alignas(16) alignas(int) int x;")
same_text("int f() __attribute__((noinline, format(printf, 1, 2)));")
same_text("[[gnu::cold]] void f();")
assert first("[[gnu::cold]] void f();").attributes[0].attributes[0].namespace == "gnu"
same_text("[[deprecated]];")
assert isinstance(first("[[deprecated]];"), S.AttributeDeclaration)
same_text(";")
assert isinstance(first(";"), S.EmptyDeclaration)
same_text("static_assert(true);\n_Static_assert(1, \"m\");")
same_text("namespace a = b;\nnamespace c = ::d::e;\nusing namespace std;\nusing std::cout;\n"
          "using enum E;")
same_text('extern "C" int x;\nextern "C++" {\n    int y;\n}')
same_text("namespace a::inline b::c {}\ninline namespace d {}\nnamespace [[deprecated]] e {}\nnamespace {}")
same_text("template void f<int>(int);\ntemplate class X<int>;\nextern template class X<long>;")
assert first("extern template class X<long>;").extern and not first("template class X<int>;").extern
with raises(ParseError, match="unsupported syntax: gnu_asm_expression"):
    CPP26.parse('int x asm("y");')
same_text('asm volatile inline goto("jmp %l0" : : : : done);')
same_text('__asm__("nop");\nasm("mov %1, %0" : "=r" (a) : [in] "r" (b) : "memory", "cc");')
print("ok")

## PRS-07 · Declarators nest inside out

In [ ]:
declarator = first("int (*f)(int);").declarators[0].declarator
assert isinstance(declarator, S.FunctionDeclarator) and isinstance(declarator.declarator, S.ParenthesizedDeclarator)
assert isinstance(declarator.declarator.declarator, S.PointerDeclarator)
same_text("int *a[3], (*b)[3], **c, &d = x, &&e = 1, *const f = 0, *g(int), (*h(int))(char);")
same_text("void (*handlers[4])(int);")
same_text("int S::*p = &S::x;\nint A::B::*q;\nint ::C::*r;")
member = first("int A::B::*q;").declarators[0].declarator
assert member.scope.qualifiers[0].spelling == "A" and member.scope.name.spelling == "B"
same_text("int x [[a]], y [[b]][3], f [[c]]();")
same_text("void f(int[*], int[static 3], int[const 4]);", standard=C23)
same_text("auto [a, b] = t;\nauto &[c, d] = u;")
same_text("void f(int, ...);\nvoid g(...);\nvoid h(int(*)(int), int (*p)[3] = nullptr);")
same_text("template <class... Ts>\nvoid f(Ts... xs, Ts &&...ys);")
same_text("template <class... Ts>\nvoid f(Ts...);")
same_text("void f() const & noexcept(true) [[a]] -> int;", "void f() const & noexcept(true) [[a]] -> int;")
same_text("void f() throw(int, char);", standard=CcppStandard(2014))
same_text("void f() requires C<T>;\nvirtual void g() override final;\nvirtual void h() = 0;")
init = first("struct S { virtual void (f)() = 0; int *p = 0; virtual int *g() = 0; int (*h)() = 0; };"
             ).specifiers[0].body.items
assert init[0].declarators[0].pure and init[1].declarators[0].initializer.value.spelling == "0"
assert init[2].declarators[0].pure and not init[3].declarators[0].pure and init[3].declarators[0].initializer
assert first("struct A { virtual ~A() = 0; };").specifiers[0].body.items[0].declarators[0].pure
with raises(ParseError, match="unsupported syntax: ms_call_modifier"):
    CPP26.parse("int (__cdecl *f)(int);")
with raises(ParseError, match="unsupported syntax: ms_call_modifier"):
    CPP26.parse("void __cdecl f() {}")
with raises(ParseError, match="unsupported syntax: ms_pointer_modifier"):
    CPP26.parse("int * __sptr p;")
with raises(ParseError, match="unsupported syntax: gnu_asm_expression"):
    CPP26.parse('void f() asm("g");')
with raises(ParseError, match="unsupported syntax: requires_clause"):
    CPP26.parse("void f(int g() requires C);")
with raises(ParseError, match="unsupported syntax: gnu_asm_expression"):
    CPP26.parse('void f() { x = asm("nop"); }')
with raises(ParseError, match="unsupported syntax: qualified_identifier"):
    CPP26.parse("auto f = [a::b] {};")
print("ok")

## PRS-08 · Statements, labels and their substatements

In [ ]:
body = first("void f() { switch (x) { case 1: a(); b(); case 2: case 3: c(); default: ; } }").body.items[0].body.items
assert [type(i).__name__ for i in body] == ["CaseStatement", "ExpressionStatement", "CaseStatement",
                                            "DefaultStatement"]
assert isinstance(body[2].statement, S.CaseStatement) and isinstance(body[2].statement.statement,
                                                                      S.ExpressionStatement)
labels = first("void f() { switch (x) { case 1: } }", CcppStandard(2023)).body.items[0].body.items
assert labels[0].statement is None
same_text("void f() {\n    if (a)\n        b();\n    else if (c) {\n        d();\n    } else\n        e();\n}")
same_text("void f() {\n    if constexpr (A) {}\n    if (int x = 1; x) {}\n    if (auto y = g())\n        h(y);\n}")
same_text("void f() {\n    while (a--)\n        continue;\n    do\n        b();\n    while (c);\n    for (;;)\n"
          "        break;\n    for (x = 0, y = 1; x < y; ++x) {}\n    for (int i = 0; i < n; ++i) {}\n"
          "    for (auto &[k, v] : m) {}\n    for (int v : {1, 2}) {}\n    for (T t = u; auto x : t) {}\n}")
same_text("void f() {\n    switch (int m = g(); m) {\n        case 1: {\n            break;\n        }\n    }\n}")
same_text("void f() {\nstart:\n    goto start;\nend:\n    int x;\n    return;\n}")
same_text("void f() {\n    try {\n        throw E();\n    } catch (const E &e) {\n        throw;\n    } catch (...) {}\n}")
same_text("void f() try {} catch (...) {}")
same_text("S::S() try : a(1) {} catch (...) {}")
same_text("struct S {\n    S() try : a(1), b{2} {} catch (...) {}\n};")
same_text("void f() {\n    [[likely]] if (a) {}\n    [[fallthrough]];\n}")
same_text("task f() {\n    co_yield 1;\n    co_return 2;\n    co_return;\n}")
with raises(ParseError, match="unsupported syntax: seh_try_statement"):
    CPP26.parse("void f() { __try {} __finally {} }")
print("ok")

## PRS-09 · Classes, enumerations and templates

In [ ]:
class_ = first("class [[a]] S final : public virtual B, [[b]] C<int>, Ds... "
               "{ public: int x : 3, y; S() : x(1), y{2}, Ds(3)... {} friend class F; friend T; friend void g(); "
               "protected: };").specifiers[0]
assert [type(a).__name__ for a in class_.attributes] == ["StandardAttributeSpecifier"]
assert [type(a).__name__ for a in first("class alignas(8) T {};").specifiers[0].attributes] == ["AlignasSpecifier"]
assert [type(a).__name__ for a in first("class __declspec(x) T {};").specifiers[0].attributes] == ["DeclspecSpecifier"]
assert class_.final and [(b.access, b.virtual, b.pack) for b in class_.bases] == [
    ("public", True, False), (None, False, False), (None, False, True)]
assert class_.bases[1].attributes and [type(i).__name__ for i in class_.body.items] == [
    "AccessSpecifier", "SimpleDeclaration", "FunctionDefinition", "SimpleDeclaration", "SimpleDeclaration",
    "SimpleDeclaration", "AccessSpecifier"]
assert class_.body.items[2].initializers[2].pack and class_.body.items[1].declarators[0].bitfield.spelling == "3"
friends = class_.body.items[3:6]
assert all(d.specifiers[0].keyword == "friend" for d in friends)
assert isinstance(friends[0].specifiers[1], S.ClassSpecifier) and isinstance(friends[1].specifiers[1],
                                                                             S.NamedTypeSpecifier)
same_text("struct S;\nunion U {\n    int i;\n    float f;\n};\nstruct {\n    int x;\n} s;")
same_text("enum E : int;\nenum class F { A, B = 2 };", "enum E : int;\nenum class F {\n    A,\n    B = 2\n};")
same_text("enum { A };", "enum {\n    A\n};")
assert first("enum { A };").specifiers[0].name is None
same_text("enum struct G : char {\n    A,\n    // comment\n    B,\n};")
same_text("template <class T, class = void, int N = 3, typename... Ts, template <class> class U = std::vector, "
          "template <class> class... Vs>\nstruct X;")
same_text("template <>\nstruct X<int> {};")
same_text("template <class T>\ntemplate <class U>\nvoid X<T>::f(U) {}")
print("ok")

## PRS-10 · Directives and comments are kept where items are listed

In [ ]:
same_text("#include <a>\n#include \"b.h\"\n#include MACRO\n#define X\n#define Y 1\n#define Z(a, b) a##b\n"
          "#define V(...) f(__VA_ARGS__)\n#define W(a, ...) a\n#pragma once\n#undef X\n#line 3 \"f\"\n#error stop")
assert isinstance(first("#include MACRO\n").macro, S.IdExpression)
assert first("#pragma\nint x;").text is None
same_text("#if defined(A) && B > 1\nint a;\n#elif !defined(C)\nint c;\n#else\nint d;\n#endif")
same_text("#ifdef A\n#elifdef B\nint b;\n#elifndef C\n#endif\n#ifndef D\n#endif", standard=CcppStandard(2023))
same_text("#if defined A\n#endif", "#if defined(A)\n#endif")
same_text("struct S {\n#ifdef A\n    int a;\n#endif\n};")
same_text("enum E {\n    A,\n#ifdef B\n    B,\n#endif\n    C\n};")
same_text("void f() {\n#if X\n    a();\n#endif\n}")
same_text("// line\n/* block */\nint x; // trailing\nvoid f() {\n    // inside\n}")
comments = parse("/* a */\n// b\r\n").items
assert [(c.block, c.text) for c in comments] == [(True, " a "), (False, " b")]
same_text("void f() {\n    switch (x) {\n        case 1: // one\n            a();\n    }\n}",
          "void f() {\n    switch (x) {\n        case 1:\n            a();\n            // one\n    }\n}")
print("ok")

## PRS-11 · Modules, `export` and `extern template`, which the pre-pass puts back

In [ ]:
unit = same_text("module;\n#include \"a.h\"\nexport module a.b:c;\nimport std;\nexport import :d;\nimport <vector>;\n"
                 "import \"e.h\";\nimport f.g:h;\nexport int x;\nexport {\n    int y;\n}\nexport namespace n {}\n"
                 "module :private;")
assert [type(i).__name__ for i in unit.items] == [
    "GlobalModuleFragment", "IncludeDirective", "ModuleDeclaration", "ImportDeclaration", "ImportDeclaration",
    "ImportDeclaration", "ImportDeclaration", "ImportDeclaration", "ExportDeclaration", "ExportDeclaration",
    "ExportDeclaration", "PrivateModuleFragment"]
assert (unit.items[2].export, unit.items[2].name, unit.items[2].partition) == (True, "a.b", "c")
assert (unit.items[4].name, unit.items[4].partition, unit.items[5].system, unit.items[6].system) == (None, "d", True, False)
same_text("export\nmodule\n    m\n;", "export module m;")
same_text('int x = module + import; // export module m;\nconst char *s = "import x;";\nint y = R"(module)";')
same_text("int f(int module);\nvoid g() {\n    import(1);\n    module.x = 2;\n    export_ = 3;\n}")
same_text("struct S {\n    template <class T>\n    void f();\n};\nextern template void S::f<int>();")
assert [type(i).__name__ for i in parse("#define M a \\\n  b\nexport int x;").items] == ["DefineDirective",
                                                                                      "ExportDeclaration"]
fails("int x; /* open", "syntax error", 1, 8)
fails("import <vector", "expected ';'", 1, 15)
fails('auto s = R"d(x', "syntax error", 1, 1)
fails('auto s = R"d', "syntax error", 1, 1)
fails("int x; export", "expected ;", 1, 14)
fails("export module m", "expected ';'", 1, 16)
fails("module :p;", "expected a module name after module", 1, 1)
fails("export module;", "expected a module name after module", 1, 8)
fails("module a.;", "expected a module name after module", 1, 1)
fails("module 3a;", "expected a module name after module", 1, 1)
fails("import;", "expected a module name after import", 1, 1)
fails('import "a.h" b;', "expected a module name after import", 1, 1)
fails("export module m [[a]];", "attributes of module declarations are not supported", 1, 8)
fails("import <a> [[x]];", "attributes of import declarations are not supported", 1, 1)
print("ok")

## PRS-12 · Errors name their line and column, counted in characters

In [ ]:
fails("int x = ;", "syntax error", 1, 7)
fails("int x", "expected ;", 1, 6)
fails("int é = 1;\nint ü", "expected ;", 2, 6)
fails("/* 🙂 */ int x", "expected ;", 1, 14)  # an astral character is one character
fails("int x = static_cast<int>(1, 2);", "static_cast takes one operand", 1, 25)
fails("auto f = [](auto x) requires C<x> {};\nconcept C = true;",
      "LambdaDeclarator.requires needs C++20, but this is C++17", 1, 12, Ccpp17)  # the first in source order
fails("void f() {\n    if consteval {}\n}", "syntax error", 2, 5)
fails("int x = a <=> b;", "BinaryExpression.operator '<=>' needs C++20, but this is C++17", 1, 9, Ccpp17)
fails("#warning w", "#warning needs C++23, but this is C++20", 1, 1, Ccpp20)
fails("int x = _Generic(1, int: 2);", "GenericSelection is not C++, but this is C++26", 1, 9)
fails("void f() throw(int);", "ThrowSpecifier with types was removed in C++17, but this is C++17", 1, 10, Ccpp17)
assert parse("register int x;", C23).items[0].specifiers[0].keyword == "register"
print("ok")

## PRS-13 · C: generic selections, compound literals and designators

In [ ]:
same_text("int x = _Generic(y, int: 1, char*: 2, default: 3);", standard=C23)
generic = initializer("int x = _Generic(y, int: 1, default: 3);", C23)
assert generic.associations[0].type is not None and generic.associations[1].type is None
same_text("struct P p = (struct P){1, 2};", standard=C23)
assert isinstance(initializer("struct P p = (struct P){1, 2};", C23), S.CompoundLiteralExpression)
same_text("int a[] = {[1] = 2, [3 ... 4] = 5, .x = 6, .y.z = 7};", standard=C23)
same_text("struct S s = {x: 1};", "struct S s = {.x = 1};", standard=C23)
same_text("_Bool b;\nint *restrict p;\n_Noreturn void f(void);\n_Atomic int a;\n_Alignas(8) int c;", standard=C23)
print("ok")